# Edge Finding Playground
Interaktive Erkundung der Kantenfindung auf den dewarped Daten.

Alle Methoden kommen aus `EDGE_METHOD_CATALOG` in `flamespread.py` — derselben Quelle, die auch die App nutzt.

In [ ]:
import sys

sys.path.insert(0, "../src")

import h5py
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import Dropdown, IntSlider, interact

from flametrack.analysis.flamespread import (
    EDGE_METHOD_CATALOG,
    calculate_edge_data,
)

In [ ]:
# ── Datei laden ───────────────────────────────────────────────────────────────
H5_PATH = (
    "../tests/_outputs/real_image_test/processed_data/real_image_test_results_RCE.h5"
)

with h5py.File(H5_PATH, "r") as f:

    def _show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: shape={obj.shape}  dtype={obj.dtype}")

    f.visititems(_show)

    key = "dewarped_data/data"
    data = f[key][()].astype(np.float32)  # (H, W, T)

H, W, T = data.shape
print(f"\nGeladene Daten: H={H}  W={W}  T={T}")
print(
    f"Wertebereich:   min={data.min():.1f}  max={data.max():.1f}  mean={data.mean():.1f}"
)

In [ ]:
# ── Methoden aus dem zentralen Katalog laden ──────────────────────────────────
#
# EDGE_METHOD_CATALOG ist die einzige Quelle der Wahrheit — die App benutzt
# exakt dieselben Spec-Objekte.  Damit ist sichergestellt, dass Notebook-
# Ergebnisse und App-Ergebnisse identisch sind.
#
# Hinweis zu Otsu:
#   calculate_edge_data verwendet Otsu, um den SUCHBEREICH pro Zeile
#   einzuschränken (welche Spalten überhaupt abgesucht werden).  Der
#   Threshold-Parameter in left/right_most_point_over_threshold bestimmt
#   dann, ab welcher Intensität ein Pixel als Kante gilt.  Beide Parameter
#   wirken zusammen – Otsu allein ersetzt den Threshold nicht.
#
# Methoden mit  use_otsu_masking=False  scannen die volle Zeile ohne Maske;
# nützlich z. B. mit threshold=117, wenn Otsu den Suchbereich zu stark
# einschränkt oder nur zur Verifikation.

print("Verfügbare Methoden:")
for spec in EDGE_METHOD_CATALOG.values():
    otsu_tag = "+ Otsu-Maske" if spec.use_otsu_masking else "ohne Otsu   "
    print(
        f"  [{otsu_tag}]  thr_image={spec.default_threshold_image:5.0f}  "
        f"{spec.short_id:<40}  {spec.display_name}"
    )

In [ ]:
# ── Hilfsfunktion ─────────────────────────────────────────────────────────────


def run_edge(frame_3d, spec, threshold):
    """Wendet calculate_edge_data mit den Einstellungen aus `spec` an.

    spec.use_otsu_masking steuert, ob Otsu den Suchbereich einschränkt.
    threshold übersteuert den Katalog-Default, damit der Slider wirkt.
    """
    fn = spec.make_fn(threshold)
    return calculate_edge_data(
        data=frame_3d,
        find_edge_point=fn,
        custom_filter=lambda x: x,
        use_otsu_masking=spec.use_otsu_masking,
    )[0]


# Display-Namen → Spec-Objekte für die Dropdown-Widgets
ALGO_OPTIONS = {spec.display_name: spec for spec in EDGE_METHOD_CATALOG.values()}

# Standardmethode und -threshold für das Widget
DEFAULT_ALGO = list(ALGO_OPTIONS.keys())[0]  # leftmost_threshold
DEFAULT_THRESHOLD = int(
    EDGE_METHOD_CATALOG["leftmost_threshold"].default_threshold_image
)  # 155

print(f"Standard-Algorithmus : {DEFAULT_ALGO}")
print(f"Standard-Threshold   : {DEFAULT_THRESHOLD}")

In [ ]:
# ── Interaktive Vorschau ──────────────────────────────────────────────────────
%matplotlib inline


def show_edge(frame_idx, threshold, algo_name):
    spec = ALGO_OPTIONS[algo_name]
    frame_3d = data[:, :, frame_idx : frame_idx + 1]  # (H, W, 1)
    frame_2d = data[:, :, frame_idx]  # (H, W)

    edges = np.array(run_edge(frame_3d, spec, threshold))
    valid = edges < W

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # --- Bild + Kante ---
    ax = axes[0]
    ax.imshow(
        frame_2d,
        cmap="plasma",
        aspect="auto",
        vmin=0,
        vmax=data.max(),
        extent=[0, W, H, 0],
    )
    ax.plot(edges[valid], np.where(valid)[0], "c-", lw=1.5, label="Kante")
    ax.plot(edges[~valid], np.where(~valid)[0], "rx", ms=3, label="nicht gefunden")
    otsu_label = "+ Otsu" if spec.use_otsu_masking else "kein Otsu"
    ax.set_title(f"Frame {frame_idx}  |  threshold={threshold}  |  {otsu_label}")
    ax.set_xlabel("Pixel (Breite)")
    ax.set_ylabel("Pixel (Höhe)")
    ax.legend(fontsize=8)

    # --- Zeilenprofil Mitte ---
    ax2 = axes[1]
    row_idx = H // 2
    row = frame_2d[row_idx]
    ax2.plot(row, color="orange", label=f"Zeile {row_idx}")
    ax2.axhline(threshold, color="red", lw=1.5, ls="--", label=f"threshold={threshold}")
    edge_pos = edges[row_idx]
    if edge_pos < W:
        ax2.axvline(edge_pos, color="cyan", lw=2, label=f"Kante @ {edge_pos}")
    ax2.set_title(f"Zeilenprofil (Zeile {row_idx})  –  via calculate_edge_data")
    ax2.set_xlabel("Pixel (Breite)")
    ax2.set_ylabel("Intensität")
    ax2.legend(fontsize=8)

    n_found = int(valid.sum())
    n_missed = int((~valid).sum())
    fig.suptitle(
        f"{spec.short_id}  |  gefunden: {n_found}/{H}  |  nicht gefunden: {n_missed}",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()


interact(
    show_edge,
    frame_idx=IntSlider(
        min=0,
        max=T - 1,
        step=1,
        value=T // 2,
        description="Frame",
        continuous_update=False,
    ),
    threshold=IntSlider(
        min=0,
        max=int(data.max()),
        step=1,
        value=DEFAULT_THRESHOLD,
        description="Threshold",
        continuous_update=False,
    ),
    algo_name=Dropdown(
        options=list(ALGO_OPTIONS.keys()), value=DEFAULT_ALGO, description="Algorithmus"
    ),
);

In [ ]:
# ── Verteilung der Kantenpositionen ──────────────────────────────────────────
def show_distribution(frame_idx, threshold, algo_name):
    spec = ALGO_OPTIONS[algo_name]
    frame_3d = data[:, :, frame_idx : frame_idx + 1]
    edges = np.array(run_edge(frame_3d, spec, threshold))
    valid = edges < W

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(edges[valid], bins=W, range=(0, W), color="cyan", edgecolor="navy")
    ax.set_title(
        f"Verteilung Kantenpositionen  |  {valid.sum()}/{H} Zeilen gefunden\n"
        f"{spec.short_id}  |  threshold={threshold}"
    )
    ax.set_xlabel("Pixel-Position")
    ax.set_ylabel("Anzahl Zeilen")
    plt.tight_layout()
    plt.show()


interact(
    show_distribution,
    frame_idx=IntSlider(
        min=0,
        max=T - 1,
        step=1,
        value=T // 2,
        description="Frame",
        continuous_update=False,
    ),
    threshold=IntSlider(
        min=0,
        max=int(data.max()),
        step=1,
        value=DEFAULT_THRESHOLD,
        description="Threshold",
        continuous_update=False,
    ),
    algo_name=Dropdown(
        options=list(ALGO_OPTIONS.keys()), value=DEFAULT_ALGO, description="Algorithmus"
    ),
);

In [ ]:
# ── Methodenvergleich: alle Algorithmen auf einmal (Frame 0) ──────────────────
#
# Zeigt alle Katalog-Methoden mit ihrem jeweiligen default_threshold_image
# nebeneinander — nützlich zum schnellen Vergleich.

COMPARE_FRAME = 0

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(
    data[:, :, COMPARE_FRAME],
    cmap="plasma",
    aspect="auto",
    vmin=0,
    vmax=data.max(),
    extent=[0, W, H, 0],
)

colors = plt.cm.tab10.colors
for color, (sid, spec) in zip(colors, EDGE_METHOD_CATALOG.items()):
    thr = spec.default_threshold_image
    edges = np.array(run_edge(data[:, :, COMPARE_FRAME : COMPARE_FRAME + 1], spec, thr))
    valid = edges < W
    label = f"{spec.short_id}  (thr={thr:.0f})"
    ax.plot(edges[valid], np.where(valid)[0], lw=1.2, color=color, label=label)

ax.set_title(f"Methodenvergleich — Frame {COMPARE_FRAME}")
ax.set_xlabel("Pixel (Breite)")
ax.set_ylabel("Pixel (Höhe)")
ax.legend(fontsize=7, loc="upper left")
plt.tight_layout()
plt.show()